# Walmart Store Sales Forecasting — LightGBM

Metric: **WMAE** (holiday weeks weighted 5×).

**Final model:** LightGBM, L1 objective, `num_leaves=127` with regularization, 2000 trees @ lr 0.03.
**Validation WMAE: 1281.58** (train 1247.75, overfit gap 33.83) on a leakage-free time-based split.

In [27]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip


## 1. Setup

Imports grouped by purpose, then the shared preprocessing module and the MLflow/DagsHub tracking connection.

In [28]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

!wget -q -O preprocessing.py "https://raw.githubusercontent.com/IrakliZerekidze/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting/main/preprocessing.py"
from preprocessing import run_pipeline, weighted_mae, THANKSGIVING, CHRISTMAS

In [29]:
%pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='izere23',
             repo_name='ML-Final-Walmart-Recruiting-Store-Sales-Forecasting',
             mlflow=True)

import mlflow
import mlflow.sklearn
mlflow.set_experiment("LightGBM_Training")

Note: you may need to restart the kernel to use updated packages.


Initialized MLflow to track repo "izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting"

Repository izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/8d957b30690b42c6a79e41cce7b15d16', creation_time=1783622687136, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1783622687136, lifecycle_stage='active', name='LightGBM_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

## 2. Data & validation split

In [30]:
train_part, valid_part, train_full, test_full = run_pipeline(
    data_dir="/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting",
    out_dir="/kaggle/working/processed",
)

# patch fill_grid's IsHoliday_x/_y collision
for df in (train_part, valid_part, test_full):
    if "IsHoliday" not in df.columns and "IsHoliday_x" in df.columns:
        df.rename(columns={"IsHoliday_x": "IsHoliday"}, inplace=True)
        df.drop(columns=["IsHoliday_y"], inplace=True)

TARGET = "Weekly_Sales"
train = train_part.dropna(subset=[TARGET]).copy()
valid = valid_part.dropna(subset=[TARGET]).copy()

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_val,   y_val   = valid.drop(columns=[TARGET]), valid[TARGET]

print(X_train.shape, X_val.shape)

Expected rows if no gaps: 428409
Actual rows: 380107
Missing (Store,Dept,Date) combos filled in: 48302
Expected rows if no gaps: 476333
Actual rows: 421570
Missing (Store,Dept,Date) combos filled in: 54763
Saved parquet files to /kaggle/working/processed
(380107, 32) (41463, 31)


## 3. Feature transformers

Each transformer is `fit` on the training fold only (all target-derived statistics are learned from train `y`),
so nothing about the validation period leaks into the features. Only the transformers used by the final model
are kept here.

- **`RichEncoder`** — smoothed mean / std / median sales at several grouping levels (Store×Dept, Dept, Store, Dept×Type).
- **`LagAdder`** — sales lagged 52 and 104 weeks (same week last year / two years ago); these look >1yr back into train, so they're clean.
- **`YoYRatioAdder`** — `lag_52 / lag_104`, i.e. year-over-year growth of the series.
- **`CalendarExtras`** — week-of-month, month/quarter-end flags, distance to nearest holiday.
- **`WalmartPreprocessor`** — drops target-derived / date columns and casts identity columns to `category`.

In [31]:
class WalmartPreprocessor(BaseEstimator, TransformerMixin):
    """Drop target-leak columns + Date, cast identity columns to category."""
    def __init__(self):
        self.drop_cols = ["Weekly_Sales_clipped", "is_negative_sales", "Date", "was_grid_filled"]
        self.cat_cols = ["Store", "Dept", "Type"]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=[c for c in self.drop_cols if c in X.columns])
        for c in self.cat_cols:
            if c in X.columns:
                X[c] = X[c].astype("category")
        if "IsHoliday" in X.columns:
            X["IsHoliday"] = X["IsHoliday"].astype(int)
        return X

In [32]:
class StoreDeptTargetEncoder(BaseEstimator, TransformerMixin):
    """Add mean historical sales per (Store, Dept), learned on TRAIN only."""
    def __init__(self, m=10):
        self.m = m

    def fit(self, X, y):
        df = X[["Store", "Dept"]].copy()
        df["y"] = np.asarray(y)
        self.global_mean_ = df["y"].mean()
        stats = df.groupby(["Store", "Dept"])["y"].agg(["mean", "count"])
        smoothed = (stats["count"] * stats["mean"] + self.m * self.global_mean_) \
                   / (stats["count"] + self.m)
        self.mapping_ = smoothed.to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        keys = list(zip(X["Store"], X["Dept"]))
        X["store_dept_te"] = [self.mapping_.get(k, self.global_mean_) for k in keys]
        return X

In [33]:
class LagAdder(BaseEstimator, TransformerMixin):
    """Add sales lagged by N weeks per (Store,Dept), looked up by calendar date."""
    def __init__(self, lags=(52,)):
        self.lags = lags

    def fit(self, X, y):
        h = X[["Store", "Dept", "Date"]].copy()
        h["sales"] = np.asarray(y)
        self.history_ = (h.drop_duplicates(["Store", "Dept", "Date"])
                          .set_index(["Store", "Dept", "Date"])["sales"])
        return self

    def transform(self, X):
        X = X.copy()
        for L in self.lags:
            lag_date = X["Date"] - pd.Timedelta(weeks=L)
            idx = pd.MultiIndex.from_arrays([X["Store"], X["Dept"], lag_date])
            X[f"lag_{L}"] = self.history_.reindex(idx).values
        return X

In [34]:
class StoreContextAdder(BaseEstimator, TransformerMixin):
    """Each dept's average share of its store's weekly sales. Train-only."""
    def fit(self, X, y):
        h = X[["Store", "Dept", "Date"]].copy()
        h["sales"] = np.asarray(y)
        store_week = h.groupby(["Store", "Date"])["sales"].sum().rename("stot")
        h = h.merge(store_week, on=["Store", "Date"], how="left")
        h["share"] = h["sales"] / h["stot"].replace(0, np.nan)
        self.dept_share_ = h.groupby(["Store", "Dept"])["share"].mean()
        self.global_share_ = h["share"].mean()
        return self

    def transform(self, X):
        X = X.copy()
        keys = list(zip(X["Store"], X["Dept"]))
        X["dept_share"] = [self.dept_share_.get(k, self.global_share_) for k in keys]
        return X

In [35]:
class RichEncoder(BaseEstimator, TransformerMixin):
    """Mean, std, median sales at multiple grouping levels. Train-only, smoothed."""
    def __init__(self, m=10):
        self.m = m
        self.levels = [["Store", "Dept"], ["Dept"], ["Store"], ["Dept", "Type"]]

    def fit(self, X, y):
        df = X.copy(); df["y"] = np.asarray(y)
        self.gmean_ = df["y"].mean()
        self.gstd_  = df["y"].std()
        self.gmed_  = df["y"].median()
        self.maps_ = {}
        for lvl in self.levels:
            if not all(c in df.columns for c in lvl):
                continue
            g = df.groupby(lvl)["y"]
            stats = g.agg(["mean", "count", "std", "median"])
            sm = (stats["count"] * stats["mean"] + self.m * self.gmean_) / (stats["count"] + self.m)
            name = "_".join(lvl)
            self.maps_[name] = (lvl, sm.to_dict(),
                                stats["std"].fillna(self.gstd_).to_dict(),
                                stats["median"].to_dict())
        return self

    def transform(self, X):
        X = X.copy()
        for name, (lvl, mean_map, std_map, med_map) in self.maps_.items():
            keys = X[lvl[0]] if len(lvl) == 1 else list(zip(*[X[c] for c in lvl]))
            keys = list(keys)
            X[f"te_mean_{name}"] = [mean_map.get(k, self.gmean_) for k in keys]
            X[f"te_std_{name}"]  = [std_map.get(k, self.gstd_) for k in keys]
            X[f"te_med_{name}"]  = [med_map.get(k, self.gmed_) for k in keys]
        return X

In [36]:
class YoYRatioAdder(BaseEstimator, TransformerMixin):
    """lag_52 / lag_104 — is the series growing or shrinking year-over-year."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        if "lag_52" in X.columns and "lag_104" in X.columns:
            X["yoy_ratio"] = X["lag_52"] / X["lag_104"].replace(0, np.nan)
        return X

In [37]:
class CalendarExtras(BaseEstimator, TransformerMixin):
    ALL_HOL = pd.to_datetime([
        "2010-02-12","2011-02-11","2012-02-10","2013-02-08",   # SuperBowl
        "2010-09-10","2011-09-09","2012-09-07","2013-09-06",   # LaborDay
        "2010-11-26","2011-11-25","2012-11-23","2013-11-29",   # Thanksgiving
        "2010-12-31","2011-12-30","2012-12-28","2013-12-27",   # Christmas
    ])
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        d = X["Date"]
        X["week_of_month"] = ((d.dt.day - 1) // 7 + 1).astype(int)
        X["is_month_end"]  = (d.dt.day >= 24).astype(int)
        X["is_quarter_end"] = ((d.dt.month.isin([3, 6, 9, 12])) & (d.dt.day >= 24)).astype(int)
        dd = d.values.astype("datetime64[D]")
        h = self.ALL_HOL.values.astype("datetime64[D]")
        diff = np.abs(h[None, :] - dd[:, None]) / np.timedelta64(1, "D")
        X["days_to_holiday"] = diff.min(axis=1)
        return X

In [38]:
class InteractionAdder(BaseEstimator, TransformerMixin):
    MD = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        md_cols = [c for c in self.MD if c in X.columns]
        if md_cols:
            X["markdown_total"] = X[md_cols].sum(axis=1)
            hol = X["IsHoliday"].astype(int) if "IsHoliday" in X.columns else 0
            X["markdown_x_holiday"] = X["markdown_total"] * hol
        if "Size" in X.columns:
            X["size_bucket"] = pd.cut(X["Size"], bins=3, labels=False)
        return X

In [39]:
class MarkdownLagAdder(BaseEstimator, TransformerMixin):
    """Last year's markdown_total + std of same-week-last-year sales history."""
    MD = ["MarkDown1","MarkDown2","MarkDown3","MarkDown4","MarkDown5"]
    def fit(self, X, y):
        h = X[["Store","Dept","Date"]].copy()
        h["sales"] = np.asarray(y)
        self.sales_hist_ = (h.drop_duplicates(["Store","Dept","Date"])
                             .set_index(["Store","Dept","Date"])["sales"])
        md = [c for c in self.MD if c in X.columns]
        m = X[["Store","Date"]].copy()
        m["md_total"] = X[md].sum(axis=1) if md else 0
        self.md_hist_ = (m.drop_duplicates(["Store","Date"])
                          .set_index(["Store","Date"])["md_total"])
        return self
    def transform(self, X):
        X = X.copy()
        lag_date = X["Date"] - pd.Timedelta(weeks=52)
        idx = pd.MultiIndex.from_arrays([X["Store"], lag_date])
        X["markdown_lag52"] = self.md_hist_.reindex(idx).values
        return X

In [40]:
class RecentLagAdder(BaseEstimator, TransformerMixin):
    """Short lags (1,2,4 wk) with history spanning the FULL provided timeline,
       so validation/test rows can look back into earlier real data.
       Pass the full labeled history at fit time."""
    def __init__(self, lags=(1, 2, 4), full_history=None):
        self.lags = lags
        self.full_history = full_history  # DataFrame with Store,Dept,Date,Weekly_Sales

    def fit(self, X, y):
        # build history from full_history if given, else from X/y
        if self.full_history is not None:
            h = self.full_history[["Store", "Dept", "Date"]].copy()
            h["sales"] = self.full_history["Weekly_Sales"].values
        else:
            h = X[["Store", "Dept", "Date"]].copy()
            h["sales"] = np.asarray(y)
        h = h.dropna(subset=["sales"]).drop_duplicates(["Store", "Dept", "Date"])
        self.history_ = h.set_index(["Store", "Dept", "Date"])["sales"]
        return self

    def transform(self, X):
        X = X.copy()
        for L in self.lags:
            lag_date = X["Date"] - pd.Timedelta(weeks=L)
            idx = pd.MultiIndex.from_arrays([X["Store"], X["Dept"], lag_date])
            X[f"rlag_{L}"] = self.history_.reindex(idx).values
        return X

In [41]:
def recursive_predict(feat, model, X_future, hist_sales, base_fn, step="recent_lags"):
    rl = feat.named_steps[step]
    hist = hist_sales[~hist_sales.index.duplicated()].copy()
    saved, preds = rl.history_, []
    try:
        for _, chunk in X_future.sort_values("Date").groupby("Date", sort=True):
            rl.history_ = hist
            resid = model.predict(feat.transform(chunk))
            out = chunk[["Store", "Dept", "Date"]].copy()
            out["pred"] = base_fn(chunk).values + resid
            preds.append(out)
            hist = pd.concat([hist, out.set_index(["Store", "Dept", "Date"])["pred"]])
    finally:
        rl.history_ = saved
    return pd.concat(preds, ignore_index=True)

In [42]:
class RollingStatsAdder(BaseEstimator, TransformerMixin):
    """Mean/std/min/max of sales over the prior `windows` weeks per (Store,Dept).
       History is train-only (fit on y); every stat uses strictly PAST weeks -> leak-safe."""
    def __init__(self, windows=(4, 8, 12)):
        self.windows = windows
    def fit(self, X, y):
        h = X[["Store", "Dept", "Date"]].copy()
        h["sales"] = np.asarray(y)
        self.hist_ = (h.dropna(subset=["sales"])
                       .drop_duplicates(["Store", "Dept", "Date"])
                       .sort_values("Date"))
        return self
    def transform(self, X):
        X = X.copy()
        base = self.hist_
        for w in self.windows:
            means, stds = [], []
            g = base.groupby(["Store", "Dept"])
            lookup = {k: v.set_index("Date")["sales"] for k, v in g}
            for s, d, dt in zip(X["Store"], X["Dept"], X["Date"]):
                ser = lookup.get((s, d))
                if ser is None:
                    means.append(np.nan); stds.append(np.nan); continue
                past = ser[ser.index < dt].tail(w)
                means.append(past.mean() if len(past) else np.nan)
                stds.append(past.std()  if len(past) > 1 else np.nan)
            X[f"roll_mean_{w}"] = means
            X[f"roll_std_{w}"]  = stds
        return X

In [43]:
from preprocessing import SUPER_BOWL, LABOR_DAY, THANKSGIVING, CHRISTMAS

class HolidayProximityAdder(BaseEstimator, TransformerMixin):
    """Signed integer weeks to the nearest occurrence of each holiday.
       Negative = weeks BEFORE, positive = AFTER, 0 = the holiday week itself.
       Captures the asymmetric pre-holiday ramp. Pure calendar (no y, no history) -> no leak."""
    def __init__(self, cap=8):
        self.cap = cap
        self.holidays = {
            "superbowl":    SUPER_BOWL,
            "labor":        LABOR_DAY,
            "thanksgiving": THANKSGIVING,
            "christmas":    CHRISTMAS,
        }
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        d = X["Date"].values.astype("datetime64[D]")
        for name, hols in self.holidays.items():
            h = np.asarray(pd.DatetimeIndex(hols).values, dtype="datetime64[D]")
            diff = (d[:, None] - h[None, :]) / np.timedelta64(7, "D")
            nearest = diff[np.arange(len(d)), np.argmin(np.abs(diff), axis=1)]
            X[f"wk_to_{name}"] = np.clip(np.round(nearest), -self.cap, self.cap).astype(int)
        return X            # <-- must be here, aligned with `d =`, OUTSIDE the for-loop

In [44]:
class DeptWeekTargetEncoder(BaseEstimator, TransformerMixin):
    """Smoothed target encoding of the (Dept, WeekOfYear) pair — captures dept-specific
       seasonality shape. Fit on train only; unseen keys fall back to the global mean. Leak-safe."""
    def __init__(self, m=10):
        self.m = m
    def fit(self, X, y):
        df = X[["Dept"]].copy()
        df["woy"] = X["Date"].dt.isocalendar().week.astype(int)
        df["y"] = np.asarray(y)
        self.global_ = df["y"].mean()
        stats = df.groupby(["Dept", "woy"])["y"].agg(["mean", "count"])
        # smoothing: pull small groups toward the global mean
        stats["enc"] = (stats["mean"] * stats["count"] + self.global_ * self.m) / (stats["count"] + self.m)
        self.map_ = stats["enc"].to_dict()
        return self
    def transform(self, X):
        X = X.copy()
        woy = X["Date"].dt.isocalendar().week.astype(int)
        keys = zip(X["Dept"], woy)
        X["te_dept_woy"] = [self.map_.get(k, self.global_) for k in keys]
        return X

In [45]:
class SeriesSeasonalBaseline(BaseEstimator, TransformerMixin):
    """Per-(Store,Dept) level x normalized Dept seasonal shape.
       level  = mean sales of that series (magnitude)
       shape  = mean of (sales / level) by (Dept, WeekOfYear), pooled over ALL stores -> generalizes
       Fit on train only -> leak-safe. Adds 3 features; model still trains on absolute sales."""
    def fit(self, X, y):
        df = X[["Store", "Dept"]].copy()
        df["woy"] = X["Date"].dt.isocalendar().week.astype(int)
        df["y"]   = np.asarray(y)
        self.g_ = df["y"].mean()
        self.level_ = df.groupby(["Store", "Dept"])["y"].mean()
        self.store_ = df.groupby("Store")["y"].mean()
        lvl = self.level_.reindex(list(zip(df["Store"], df["Dept"]))).values
        df["ratio"] = df["y"] / np.where(lvl > 0, lvl, self.g_)
        self.shape_ = df.groupby(["Dept", "woy"])["ratio"].mean()   # normalized dept seasonality
        return self
    def transform(self, X):
        X = X.copy()
        woy = X["Date"].dt.isocalendar().week.astype(int)
        lvl = pd.Series(self.level_.reindex(list(zip(X["Store"], X["Dept"]))).values, index=X.index)
        lvl = lvl.fillna(X["Store"].map(self.store_)).fillna(self.g_)
        shp = pd.Series(self.shape_.reindex(list(zip(X["Dept"], woy))).values, index=X.index).fillna(1.0)
        X["sd_level"]        = lvl
        X["dept_seas_idx"]   = shp
        X["series_baseline"] = lvl * shp
        return X

## 4. Final model & pipeline

**Objective `regression_l1`** — WMAE is an L1 (absolute-error) metric, so L1 training matches it directly and
avoids chasing the huge-sales outliers that L2 over-weights.

**`num_leaves=127` with regularization** — a large tree finds a lower validation score, but only pays off when
paired with `min_child_samples=200` + `reg_lambda=10` to stop it memorizing. Un-regularized, this config overfits
badly (gap ~330); regularized, the gap collapses to ~30 while keeping the low score.

**2000 trees @ lr 0.03** — slower learning with more rounds extracts a little more signal without reopening the gap.

This is the configuration that produced the best trustworthy result (val 1281.58, gap 33.83).

In [46]:
model = LGBMRegressor(
    n_estimators=2000, learning_rate=0.03, num_leaves=127,
    subsample=0.8, colsample_bytree=0.8,
    min_child_samples=200, reg_lambda=10.0,
    random_state=42, n_jobs=-1, verbose=-1,
    objective="regression_l1",
)

pipeline = Pipeline([
    ("rich_encoding", RichEncoder(m=5)),
    ("lags",          LagAdder(lags=(52, 104))),
    ("yoy",           YoYRatioAdder()),
    ("calendar",      CalendarExtras()),
    ("preprocessing", WalmartPreprocessor()),
    ("model", model),
])

## 5. Train, evaluate & log

In [47]:
with mlflow.start_run(run_name="LightGBM_leaves127_n2000_lr03_FINAL"):
    pipeline.fit(X_train, y_train)
    train_pred = pipeline.predict(X_train)
    val_pred   = pipeline.predict(X_val)

    train_wmae = weighted_mae(y_train.values, train_pred, X_train["IsHoliday"].values)
    val_wmae   = weighted_mae(y_val.values,   val_pred,   X_val["IsHoliday"].values)
    val_mae    = float(np.mean(np.abs(y_val.values - val_pred)))
    val_rmse   = float(np.sqrt(np.mean((y_val.values - val_pred) ** 2)))
    overfit_gap = val_wmae - train_wmae

    mlflow.log_param("model", "LightGBM")
    mlflow.log_param("baseline_type", "leaves127_n2000_lr03_final")
    mlflow.log_param("features_added", "richenc_m5,lags52_104,yoy,calendar | obj=L1")
    mlflow.log_params(model.get_params())
    mlflow.log_metric("train_wmae", train_wmae)
    mlflow.log_metric("val_wmae", val_wmae)
    mlflow.log_metric("overfit_gap", overfit_gap)
    mlflow.log_metric("val_mae", val_mae)
    mlflow.log_metric("val_rmse", val_rmse)

    print(f"train WMAE : {train_wmae:.2f}")
    print(f"val   WMAE : {val_wmae:.2f}")
    print(f"overfit gap: {overfit_gap:.2f}")
    print(f"val MAE    : {val_mae:.2f}")
    print(f"val RMSE   : {val_rmse:.2f}")

train WMAE : 1247.75
val   WMAE : 1281.58
overfit gap: 33.83
val MAE    : 1270.70
val RMSE   : 2742.23
🏃 View run LightGBM_leaves127_n2000_lr03_FINAL at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/3/runs/655f261be96c4922bb98007844bc901b
🧪 View experiment at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/3


## 6. Experiment log & findings

All runs use the same time-based split and WMAE, so the numbers are directly comparable.

| Run | val WMAE | gap | verdict |
|---|---|---|---|
| `RecentLags` (short lags fed **actual** history) | 1184.60 | 23.9 | ❌ **leakage** — removed |
| `RecentLags_recursive` (lags fed from predictions) | 1366.55 | 21.5 | honest version; short lags add nothing |
| Clean baseline (RichEnc + lag52/104 + YoY + calendar) | 1351.65 | 90.8 | reference |
| + L1 objective | 1340.29 | ~-141* | keep (metric-aligned) |
| + rolling means/std (4/8/12 wk) | 1497.03 | 323 | ❌ overfits (recent-level memorization) |
| + signed weeks-to-holiday | 1360.52 | 97 | ❌ redundant with existing calendar |
| + (Dept, WeekOfYear) target encoding | 1388.39 | 199 | ❌ overfits seasonality |
| + per-series level × seasonal shape | 1429.20 | 105 | ❌ redundant with lag_52 |
| holiday sample-weight ×5 | 1385.87 | 164 | ❌ window has ~no holidays |
| `leaves127` (lr 0.05, unregularized) | 1310.64 | 330 | best raw number, but overfit |
| `leaves127` + regularization | 1305.59 | 6.1 | keep — gap fixed |
| **`leaves127` + reg + 2000 trees @ lr 0.03** | **1281.58** | **33.8** | ✅ **final model** |
| leaves127 + mcs 100 | 1291.08 | 49.8 | worse |
| leaves127 + 3000 trees @ lr 0.02 | 1293.16 | 46.9 | worse — past the optimum |

\* L1 gap is negative because train WMAE > val WMAE (L1 stops chasing train outliers); not overfitting.

**Narrative.** A validation leak in the short-lag feature was inflating the score to 1184; removing it and
re-running the identical pipeline honestly (recursive lags) scored 1366 with the same tiny gap — confirming the
low gap was the leak signature, not a good fit. From a clean 1351 baseline, five feature families were each ruled
out with a mechanism (recent-level → overfit, calendar → redundant, interaction/seasonal → overfit/redundant),
showing the base feature set is **saturated on this window**. The real gains came from matching the loss to the
metric (L1) and from regularized capacity tuning, which earned a trustworthy **1281.58 at a 34-point gap**.

**Context.** On this no-holiday window the number reads low; public GBM solutions for this competition score
~2000–3200 WMAE on the actual Kaggle test set (winner ~2300 with a time-series ensemble), so a global GBM's
realistic leaderboard range is well above this local figure.

# Registering

In [49]:
from mlflow.models.signature import infer_signature

# --- train on ALL labeled data ---
X_all = train_full.dropna(subset=[TARGET]).drop(columns=[TARGET])
y_all = train_full.dropna(subset=[TARGET])[TARGET]

final_model = LGBMRegressor(
    n_estimators=2000, learning_rate=0.03, num_leaves=127,
    subsample=0.8, colsample_bytree=0.8,
    min_child_samples=200, reg_lambda=10.0,
    random_state=42, n_jobs=-1, verbose=-1,
    objective="regression_l1",
)
final_pipe = Pipeline([
    ("rich_encoding", RichEncoder(m=5)),
    ("lags",          LagAdder(lags=(52, 104))),
    ("yoy",           YoYRatioAdder()),
    ("calendar",      CalendarExtras()),
    ("preprocessing", WalmartPreprocessor()),
    ("model", final_model),
])

with mlflow.start_run(run_name="LightGBM_FINAL_registered") as run:
    final_pipe.fit(X_all, y_all)

    # predict on the unlabeled test set (no WMAE possible — test.csv has no Weekly_Sales)
    test_pred = np.clip(final_pipe.predict(test_full), 0, None)

    submission = pd.DataFrame({
        "Id": (test_full["Store"].astype(str) + "_" + test_full["Dept"].astype(str)
               + "_" + test_full["Date"].dt.strftime("%Y-%m-%d")),
        "Weekly_Sales": test_pred,
    })
    submission.to_csv("/kaggle/working/submission.csv", index=False)

    # log training-fold metrics (the honest, reportable numbers) + register the model
    val_pred = final_pipe.predict(X_val)
    mlflow.log_params(final_model.get_params())
    mlflow.log_param("trained_on", "all_labeled_data")
    mlflow.log_metric("val_wmae_holdout", weighted_mae(y_val.values, val_pred, X_val["IsHoliday"].values))
    mlflow.log_metric("n_test_predictions", len(submission))

    sig = infer_signature(X_all, final_pipe.predict(X_all))
    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="model",
        signature=sig,
        serialization_format="cloudpickle",          # <-- skops can't serialize custom classes; use cloudpickle
        registered_model_name="walmart_lightgbm_final",
    )

print("test rows predicted:", len(submission))
print("saved: /kaggle/working/submission.csv")
print(submission.head())

/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/07/11 09:57:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 09:57:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution becaus

🏃 View run LightGBM_FINAL_registered at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/3/runs/8aab7ddea182407790b0a6d7d605edb3
🧪 View experiment at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/3
test rows predicted: 115064
saved: /kaggle/working/submission.csv
               Id  Weekly_Sales
0  1_1_2012-11-02  36248.376788
1  1_1_2012-11-09  20730.285548
2  1_1_2012-11-16  20784.976599
3  1_1_2012-11-23  20901.472016
4  1_1_2012-11-30  26518.492035
